# Step 6: memory ladder for Gemma (BF16)

Which context budget can Gemma actually run? Rising budgets from 7,500 to 12,000 characters, full
400-token answers, stops at the first failure. About 20 min. It stops early, with a plain message,
if the hardware is wrong.

Before running: **Accelerator = GPU T4 x2**, **Internet = on**, and add your private "ptlegal-chunks"
dataset under *Input*. Then *Run All* and keep the tab open.

Paste back to Claude the text printed at the end of cell 4.

In [ ]:
# 1. Setup (about 3 min): latest code from GitHub
%cd /kaggle/working
!pip -q install -U "transformers>=5.10" accelerate huggingface_hub
!rm -rf pt-legal-rag-eval && git clone -q -b step-6-runner https://github.com/cardohso/pt-legal-rag-eval.git
%cd pt-legal-rag-eval
!pip -q install -e .
!git rev-parse HEAD

In [ ]:
# 2. Hardware gate: stops here, before any download, unless this session has two T4 GPUs
import subprocess
import sys

check = subprocess.run(
    [sys.executable, "-m", "ptlegal_eval.gpucheck"], capture_output=True, text=True
)
print(check.stdout + check.stderr)
if check.returncode != 0:
    raise RuntimeError(
        "Stopped before any download. Fix what is printed above (wrong GPU, or the setup "
        "failed: check Internet = on), then Run All again."
    )

In [ ]:
# 3. Find the chunks file you added as input (searches any depth)
import glob
import os

found = glob.glob("/kaggle/input/**/chunks.jsonl", recursive=True)
print(found)
if len(found) != 1:
    for root, _dirs, files in os.walk("/kaggle/input"):
        print(root, files[:5])
    raise RuntimeError("expected one chunks.jsonl under /kaggle/input: add the dataset as Input")
CHUNKS = found[0]

In [ ]:
# 4. The memory ladder (loads Gemma once, then one step per budget, about 2 min each)
!python -m ptlegal_eval.memladder --model gemma --chunks {CHUNKS} --out /kaggle/working/memladder

In [ ]:
# 5. One zip to download from the Output panel
!cd /kaggle/working && zip -rq memladder.zip memladder && ls -la memladder.zip memladder